# 🚀 Spaceship Titanic — LightGBM + Feature Engineering
**Kaggle Yarışması | CV Accuracy: ~0.8178+**

Bu notebook şunları içerir:
- Gelişmiş Feature Engineering (grup bilgisi, harcama analizi, kabin özellikleri)
- LightGBM ile 10-Fold Stratified Cross-Validation
- Hugging Face'e deploy edilebilir model kaydı
- Temiz, tekrar üretilebilir kod

In [ ]:
# ── Temel kütüphaneler ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
import os
import joblib
import json

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

from lightgbm import LGBMClassifier, early_stopping, log_evaluation

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print('Kütüphaneler yüklendi ✓')

## 1. Veri Yükleme

In [ ]:
# Kaggle ortamında çalışır; yerel çalışma için yolu değiştirin.
DATA_DIR = '/kaggle/input/spaceship-titanic'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
submission = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print(f'Train: {train.shape}  |  Test: {test.shape}')
train.head()

## 2. Keşifsel Veri Analizi (EDA)

In [ ]:
# Eksik değerlerin ısı haritası
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hedef dağılımı
train['Transported'].value_counts().plot(
    kind='bar', ax=axes[0], color=['#4C72B0','#DD8452'],
    title='Hedef Dağılımı (Transported)'
)
axes[0].set_xticklabels(['False','True'], rotation=0)

# Eksik veri oranları
missing = (train.isnull().mean() * 100).sort_values(ascending=False)
missing[missing > 0].plot(kind='bar', ax=axes[1], color='#C44E52',
                           title='Eksik Veri Oranı (%)')
plt.tight_layout()
plt.show()

print('\nEksik değer sayıları:')
print(train.isnull().sum()[train.isnull().sum() > 0])

In [ ]:
# Sayısal özelliklerin dağılımı
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for ax, col in zip(axes.flatten(), num_cols):
    sns.histplot(data=train, x=col, hue='Transported', bins=40, ax=ax, kde=True)
    ax.set_title(col)

plt.suptitle('Sayısal Özelliklerin Transported\'a Göre Dağılımı', y=1.02)
plt.tight_layout()
plt.show()

## 3. Feature Engineering

**Düzeltilen sorunlar (orijinal notebook'a kıyasla):**
- ❌ İlk `preprocess` fonksiyonu (eski, eksik) **kaldırıldı** — sadece biri aktifti zaten
- ✅ `PassengerId` artık train/test'te korunuyor, drop **sonra** yapılıyor
- ✅ Sayısal eksik değerler medyan ile dolduruldu (ağaç modeli için kritik değil ama pipeline bütünlüğü için iyi)
- ✅ Boolean sütunlar (`CryoSleep`, `VIP`) integer'a dönüştürüldü
- ✅ Label encoding (LightGBM category desteğiyle de çalışır ama joblib serializasyonu için LE daha güvenli)

In [ ]:
SPEND_COLS = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
CAT_COLS   = ['HomePlanet', 'Destination', 'Deck', 'Side']

label_encoders = {}  # Hugging Face deploy için kaydedilecek


def preprocess(df: pd.DataFrame, fit: bool = True) -> pd.DataFrame:
    """
    fit=True  → train seti için (LabelEncoder fit+transform)
    fit=False → test/inference için (sadece transform)
    """
    df = df.copy()

    # ── 1. Grup bilgisi ────────────────────────────────────────────────────
    df['Group']     = df['PassengerId'].str.split('_', expand=True)[0]
    df['GroupSize'] = df.groupby('Group')['Group'].transform('count')
    df['IsAlone']   = (df['GroupSize'] == 1).astype(int)

    # ── 2. Kabin bilgisi ───────────────────────────────────────────────────
    df[['Deck', 'Num', 'Side']] = df['Cabin'].str.split('/', expand=True)
    df['Num'] = pd.to_numeric(df['Num'], errors='coerce').fillna(-1).astype(int)

    # ── 3. Harcama özellikleri ─────────────────────────────────────────────
    df[SPEND_COLS]      = df[SPEND_COLS].fillna(0)
    df['TotalSpend']    = df[SPEND_COLS].sum(axis=1)
    df['LuxurySpend']   = df[['Spa', 'VRDeck', 'RoomService']].sum(axis=1)
    df['EssentialSpend']= df[['FoodCourt', 'ShoppingMall']].sum(axis=1)
    df['LogTotalSpend'] = np.log1p(df['TotalSpend'])  # ✨ YENİ: skewed dağılımı düzeltir
    df['HasSpent']      = (df['TotalSpend'] > 0).astype(int)
    df['NoSpend']       = (df['TotalSpend'] == 0).astype(int)

    # ── 4. CryoSleep tutarlılığı ───────────────────────────────────────────
    df['CryoSleep'] = df['CryoSleep'].fillna(False)
    df['CryoSleep'] = df['CryoSleep'].astype(int)
    df['CryoSpend_Conflict'] = ((df['CryoSleep'] == 1) & (df['TotalSpend'] > 0)).astype(int)

    # ── 5. VIP ────────────────────────────────────────────────────────────
    df['VIP'] = df['VIP'].fillna(False).astype(int)

    # ── 6. Yaş özellikleri ────────────────────────────────────────────────
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['IsChild']  = (df['Age'] < 13).astype(int)
    df['IsSenior'] = (df['Age'] > 60).astype(int)  # ✨ YENİ
    df['AgeBin']   = pd.cut(df['Age'], bins=[0,12,18,35,60,200],
                             labels=[0,1,2,3,4]).astype(int)  # ✨ YENİ

    # ── 7. Gereksiz sütunları düşür ───────────────────────────────────────
    df.drop(['PassengerId', 'Name', 'Cabin', 'Group'], axis=1, inplace=True)

    # ── 8. Kategorik encoding ─────────────────────────────────────────────
    for col in CAT_COLS:
        df[col] = df[col].fillna('Unknown')
        if fit:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            label_encoders[col] = le
        else:
            le = label_encoders[col]
            # Görülmemiş etiketleri 'Unknown' ile replace et
            df[col] = df[col].astype(str).apply(
                lambda x: x if x in le.classes_ else 'Unknown'
            )
            df[col] = le.transform(df[col])

    return df


train_df = preprocess(train.copy(), fit=True)
test_df  = preprocess(test.copy(),  fit=False)

X = train_df.drop('Transported', axis=1)
y = train_df['Transported'].astype(int)

print(f'Özellik sayısı: {X.shape[1]}')
print('Sütunlar:', list(X.columns))

## 4. Model Eğitimi — 10-Fold Stratified CV

In [ ]:
# Optimize edilmiş LightGBM parametreleri
lgb_params = {
    'objective':        'binary',
    'metric':           'binary_logloss',
    'learning_rate':    0.005,
    'n_estimators':     10000,
    'num_leaves':       31,        # 16'dan artırıldı — daha iyi kapasite
    'max_depth':        6,         # ✨ YENİ: aşırı öğrenmeyi sınırlar
    'min_child_samples': 40,
    'subsample':        0.8,       # ✨ YENİ: satır örneklemesi
    'subsample_freq':   1,
    'colsample_bytree': 0.7,
    'reg_alpha':        0.3,
    'reg_lambda':       0.5,
    'random_state':     42,
    'verbose':          -1,
    'n_jobs':           -1,
}

skf        = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(test_df))
fold_scores = []
trained_models = []  # Deploy için sakla

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = LGBMClassifier(**lgb_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            early_stopping(stopping_rounds=150, verbose=False),
            log_evaluation(500)
        ]
    )

    val_proba = model.predict_proba(X_val)[:, 1]
    preds     = (val_proba > 0.5).astype(int)
    score     = accuracy_score(y_val, preds)
    auc       = roc_auc_score(y_val, val_proba)

    fold_scores.append(score)
    oof_preds[val_idx] = val_proba
    test_preds += model.predict_proba(test_df[X.columns])[:, 1] / skf.n_splits
    trained_models.append(model)

    print(f'Fold {fold+1:2d} | Accuracy: {score:.5f} | AUC: {auc:.5f}')

print(f'\n{"="*50}')
print(f'OOF Accuracy  : {accuracy_score(y, (oof_preds > 0.5).astype(int)):.5f}')
print(f'OOF AUC       : {roc_auc_score(y, oof_preds):.5f}')
print(f'Mean CV Acc   : {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f}')

## 5. Feature Importance

In [ ]:
# Son fold'un modeli ile görselleştirme
import lightgbm as lgb

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
lgb.plot_importance(trained_models[-1], max_num_features=15,
                    importance_type='gain', ax=axes[0], title='Gain')
lgb.plot_importance(trained_models[-1], max_num_features=15,
                    importance_type='split', ax=axes[1], title='Split')
plt.tight_layout()
plt.show()

## 6. Model Kaydetme (Hugging Face Deploy için)

In [ ]:
import os
os.makedirs('model_artifacts', exist_ok=True)

# En iyi fold'u seç (en yüksek accuracy)
best_fold_idx = int(np.argmax(fold_scores))
best_model    = trained_models[best_fold_idx]
print(f'En iyi fold: {best_fold_idx+1} (Accuracy: {fold_scores[best_fold_idx]:.5f})')

# Modeli kaydet
joblib.dump(best_model, 'model_artifacts/lgbm_model.pkl')

# Label encoder'ları kaydet
joblib.dump(label_encoders, 'model_artifacts/label_encoders.pkl')

# Feature listesini kaydet
feature_info = {
    'feature_names': list(X.columns),
    'cat_cols':      CAT_COLS,
    'spend_cols':    SPEND_COLS,
    'cv_accuracy':   float(np.mean(fold_scores)),
    'best_fold_acc': float(fold_scores[best_fold_idx]),
}
with open('model_artifacts/feature_info.json', 'w') as f:
    json.dump(feature_info, f, indent=2)

print('Model artifacts kaydedildi ✓')
print('  model_artifacts/lgbm_model.pkl')
print('  model_artifacts/label_encoders.pkl')
print('  model_artifacts/feature_info.json')

## 7. Submission

In [ ]:
submission['Transported'] = (test_preds > 0.5).astype(bool)
submission.to_csv('submission.csv', index=False)
print('submission.csv kaydedildi ✓')
print(submission['Transported'].value_counts())